In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

from dotenv import load_dotenv
import os

In [3]:
load_dotenv()
#verifying that correct variables are loaded
print("OLLAMA_HOST:",os.getenv("OLLAMA_BASE_URL"))
print("OLLAMA_MODEL:",os.getenv("OLLAMA_MODEL"))

#pdf loading and splitting to chunks
loader=PyPDFLoader("Documents/file.pdf")
docs=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks=splitter.split_documents(docs)
print("Total chunks:", len(chunks))


OLLAMA_HOST: http://localhost:11434
OLLAMA_MODEL: llama3
Total chunks: 17


In [4]:
#checking the chunks
print(chunks[0])

page_content='SNMPSNMPSNMPSNMP
Simple Network Management System
By: Soumya ShradhaBy: Soumya ShradhaBy: Soumya ShradhaBy: Soumya Shradha' metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-03-25T01:53:24+00:00', 'title': 'SNMP', 'moddate': '2026-03-25T01:53:19+00:00', 'keywords': 'DAHE2K0lH5o,BAG7UR7851Y,0', 'author': 'Soumya Shradha', 'source': 'Documents/file.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1'}


In [5]:
#creating embedding objects
embeddings=OllamaEmbeddings(model="nomic-embed-text",base_url=os.getenv("OLLAMA_BASE_URL"))
#store it into vector database
vectorstore=Chroma.from_documents(documents=chunks,embedding=embeddings,
                                  persist_directory=os.getenv("VECTORSTORE_DIR"))
print("Vectorstore created successfully!!!")

Vectorstore created successfully!!!


In [6]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

In [7]:
system_prompt=("You are an assistant for question-answering only."
               "Use only the retrieved chunks as context to answer the question,"
               "If you don't know the answer, say that "
               "I do not know."
               "\n\nContext:\n{context}"
               )
prompt=ChatPromptTemplate.from_messages([("system",system_prompt),("human","{input}"),])
               

In [8]:
vectorstore=Chroma(persist_directory=os.getenv("VECTORSTORE_DIR"),embedding_function=embeddings)
retriever=vectorstore.as_retriever(search_kwargs={"k":3})

In [13]:
llm=ChatOllama(model=os.getenv("OLLAMA_MODEL"),base_url=os.getenv("OLLAMA_BASE_URL"),num_gpu=5,num_ctx=4098)

def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

rag_chain=({"context":retriever | format_docs,"input":RunnablePassthrough()}
           |prompt
           |llm
           |StrOutputParser())

In [14]:
question="What is SNMP?"
answer=rag_chain.invoke(question)
print(answer)

SNMP stands for Simple Network Management Protocol. It is a widely used protocol for network management that provides a standardized framework for monitoring and managing network-connected devices in IP networks, such as routers, switches, servers, printers, firewalls, and load balancers.
